# Copyright (c) 2026 hidemi-k / Licensed under the MIT License

# マルチレイヤ障害診断エージェント

## 🔍 テーマ3: 専門特化型マルチエージェント × 自動診断（マルチベンダー対応版 v5）

### 役割分担の原則
```
【YAMLが担う部分（手順書由来）】
  VENDOR_KEY（手順書キー）: 運用チームが機種ごとに自由に命名・定義
  実行コマンド定義        : 機種・用途別（何を確認するか）
  recovery_hints         : OS固有のCLI記法
  purpose                : LLMへのプロンプト文脈

【Netmikoが担う部分】
  NETMIKO_DRIVER         : SSHでどう繋ぐか（Netmikoが決める・変更不可）

【LLMが担う部分】
  フロー遷移判断          : l2 / l3 / full の選択
  コマンド出力の解釈      : 非構造化テキストから異常を特定
  整合性チェック          : Self-Correction
  診断レポート生成        : 根本原因・推奨アクション

【VENDOR_KEY と NETMIKO_DRIVER の関係】
  1つの NETMIKO_DRIVER が複数の VENDOR_KEY に対応できる

  juniper_junos（driver）→ juniper_ex        （EXシリーズ・L2スイッチ）
                         → juniper_qfx       （QFXシリーズ・ファブリック）
                         → juniper_ptx_evo   （PTXシリーズ・コアルータ）

  cisco_ios（driver）    → cisco_catalyst     （Catalystスイッチ）
                         → cisco_asr          （ASRルータ）
                         → cisco_isr          （ISRルータ）
```


## 📦 ライブラリのインストール

In [1]:
#!pip install "agent-framework==1.0.0rc5" netmiko --quiet
print("✅ インストール完了")


✅ インストール完了


## 📦 インポート

In [2]:
import asyncio
import os
import configparser
from typing import List, Dict, Any, Optional
from datetime import datetime

from agent_framework import Agent, Message
from agent_framework.openai import OpenAIChatClient

print("✅ インポート完了")


✅ インポート完了


## 🔑 APIキー設定

In [3]:
config = configparser.ConfigParser()
config_paths = [
    '/home/admin/config/config.ini',
    './config.ini',
    os.path.expanduser('~/config/config.ini')
]

GROQ_API_KEY = None
for path in config_paths:
    if os.path.exists(path):
        config.read(path)
        if 'GROQ' in config and 'GROQ_API_KEY' in config['GROQ']:
            GROQ_API_KEY = config['GROQ']['GROQ_API_KEY']
            print(f"✅ config.ini から読み込みました ({path})")
            break

if not GROQ_API_KEY:
    GROQ_API_KEY = os.getenv('GROQ_API_KEY')
    if GROQ_API_KEY:
        print("✅ 環境変数から読み込みました")

if not GROQ_API_KEY:
    GROQ_API_KEY = "your-groq-api-key-here"
    print("⚠️  GROQ_API_KEY を設定してください")
else:
    print(f"APIキー: {GROQ_API_KEY[:4]}... （設定済み）")


✅ config.ini から読み込みました (/home/admin/config/config.ini)
APIキー: gsk_... （設定済み）


## 🤖 クライアントファクトリ

In [4]:
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
DEFAULT_MODEL  = "llama-3.3-70b-versatile"

def make_client(model_id: str = DEFAULT_MODEL) -> OpenAIChatClient:
    return OpenAIChatClient(
        model_id=model_id,
        api_key=GROQ_API_KEY,
        base_url=GROQ_BASE_URL,
    )

print("✅ make_client() 定義完了")
print(f"   モデル: {DEFAULT_MODEL}")


✅ make_client() 定義完了
   モデル: llama-3.3-70b-versatile


## ⚙️ 実行モード・デバイス設定

**2つのキーを使い分けることがマルチベンダー対応の核心です。**

| パラメータ | 用途 | 誰が決める |
|-----------|------|----------|
| `NETMIKO_DRIVER` | SSH接続ドライバ（どう繋ぐか） | Netmikoの仕様（変更不可） |
| `VENDOR_KEY` | コマンド辞書のキー（何を確認するか） | 運用チームが手順書から定義 |

同じ `juniper_junos` ドライバでも、PTXとEXではコマンドが違う → `VENDOR_KEY` で区別します。


In [5]:
# ============================================================
# ✏️ 実行モードとデバイス情報を設定してください
# ============================================================
EXECUTE_MODE = "netmiko"  # "netmiko"（実機） or "mock"（テスト用）

# Netmiko 接続情報
DEVICE_CONFIG = {
    "ip":       "172.20.100.21",
    "port":     "22",
    "username": "admin",
    "password": "YOUR_PASSWORD_HERE",
}

# ✏️ NETMIKO_DRIVER: SSH接続ドライバ（Netmikoが対応しているものを指定）
#    Netmiko対応 Juniper: "juniper_junos"（PTX/EX/QFX/MX すべてこれ）
#    Netmiko対応 Cisco  : "cisco_ios" / "cisco_nxos" / "cisco_xe"
#    Netmiko対応 Arista : "arista_eos"
NETMIKO_DRIVER = "juniper_junos"

# ✏️ VENDOR_KEY: VENDOR_COMMANDS 辞書のキー（運用手順書から定義）
#    Netmiko の device_type とは独立。同じドライバで複数機種を区別できる。
#    例: "juniper_ptx_evo" / "juniper_ex" / "cisco_catalyst" など
VENDOR_KEY = "juniper_ptx_evo"

print(f"✅ 実行モード    : {EXECUTE_MODE}")
print(f"   Netmikoドライバ: {NETMIKO_DRIVER}  （SSH接続用）")
print(f"   VENDOR_KEY    : {VENDOR_KEY}  （コマンド辞書キー / 手順書由来）")
if EXECUTE_MODE == "netmiko":
    print(f"   接続先        : {DEVICE_CONFIG['ip']}:{DEVICE_CONFIG['port']}")


✅ 実行モード    : netmiko
   Netmikoドライバ: juniper_junos  （SSH接続用）
   VENDOR_KEY    : juniper_ptx_evo  （コマンド辞書キー / 手順書由来）
   接続先        : 172.20.100.21:22


## 📖 VENDOR_COMMANDS：手順書をYAML化したコマンド辞書

### 設計思想
- **`VENDOR_KEY`**: 運用チームが機種の用途・型番をもとに自由に命名する
- **Netmiko の `device_type` とは独立**: 同じドライバでも機種が違えば別キーで定義
- **`purpose`**: LLM分析プロンプトへの文脈注入（何を見ればよいか・空の場合の意味も含める）
- **`recovery_hints`**: OS固有のCLI記法（レポートのコマンド例に使用）
- **新機種追加**: このセルにエントリを追加するだけ。コードは変更不要

### キー命名の考え方
```
juniper_ex        ← Juniper EXシリーズ（L2スイッチ）
juniper_qfx       ← Juniper QFXシリーズ（データセンターファブリック）
juniper_ptx_evo   ← Juniper PTXシリーズ（コアルータ / Junos EVO）
juniper_mx        ← Juniper MXシリーズ（エッジルータ）
cisco_catalyst    ← Cisco Catalystシリーズ（L2スイッチ）
cisco_asr         ← Cisco ASRシリーズ（エッジルータ）
```


In [6]:
# ============================================================
# ✏️ 手順書をYAML化したコマンド辞書
#    VENDOR_KEY は運用チームが自由に命名する（Netmikoのdevice_typeと無関係）
#    新機種追加: エントリを追加するだけ。コードは変更不要。
# ============================================================
VENDOR_COMMANDS = {

    # ── Juniper EX シリーズ（L2スイッチ）────────────────────
    # Netmiko driver: juniper_junos
    "juniper_ex": {
        "os_label": "Juniper EX Series (L2 Switch)",
        "l2": [
            {"cmd": "show interfaces",
             "purpose": "物理インターフェースの Up/Down 状態・エラーカウンタを確認"},
            {"cmd": "show ethernet-switching table",
             "purpose": "MAC アドレステーブルの学習状態・エントリ欠落を確認"},
        ],
        "l3": [
            {"cmd": "show route",
             "purpose": "ルーティングテーブル・Inactive ルートを確認"},
            {"cmd": "show arp",
             "purpose": "ARP テーブルの解決状態・エントリ欠落を確認"},
        ],
        "recovery_hints": {
            "interface_down": "delete interfaces {ifname} disable  または  set interfaces {ifname} enable",
            "route_inactive": "show ospf neighbor / show bgp summary でプロトコル状態を確認",
            "arp_missing":    "clear arp  でARPキャッシュをクリア後、再確認",
        },
    },

    # ── Juniper PTX シリーズ（コアルータ / Junos EVO）────────
    # Netmiko driver: juniper_junos（PTX専用ドライバは存在しないため共用）
    # L2スイッチング非対応機種のため l2 はインターフェース確認のみ
    "juniper_ptx_evo": {
        "os_label": "Juniper PTX Series (Core Router / Junos EVO)",
        "l2": [
            {"cmd": "show interfaces",
             "purpose": "物理インターフェースの Up/Down 状態・エラーカウンタ・光レベルを確認"},
        ],
        "l3": [
            {"cmd": "show route",
             "purpose": "ルーティングテーブル全体・Inactive ルート・next-hop を確認"},
            {"cmd": "show arp",
             "purpose": "ARP テーブルの解決状態を確認。出力が空の場合は直接接続セグメントが存在しないことを意味し、コアルータでは正常。エントリがある場合は IP/MAC の対応関係を確認する"},
            {"cmd": "show bgp summary",
             "purpose": "BGP ピアの Up/Down・受信ルート数を確認（コアルータの主要プロトコル）"},
            {"cmd": "show ospf neighbor",
             "purpose": "OSPF ネイバーの状態・DR/BDR を確認"},
        ],
        "recovery_hints": {
            "interface_down": "set interfaces {ifname} enable  /  delete interfaces {ifname} disable",
            "bgp_peer_down":  "show bgp neighbor {peer} detail で詳細確認 → clear bgp neighbor {peer}",
            "route_inactive": "show route {prefix} detail で詳細確認 → ルーティングポリシーを見直す",
            "ospf_down":      "show ospf interface / show ospf neighbor でネイバー確認 → インターフェース設定を見直す",
        },
    },

    # ── Juniper MX シリーズ（エッジルータ）──────────────────
    # Netmiko driver: juniper_junos
    "juniper_mx": {
        "os_label": "Juniper MX Series (Edge Router)",
        "l2": [
            {"cmd": "show interfaces",
             "purpose": "物理インターフェースの Up/Down 状態・エラーカウンタを確認"},
            {"cmd": "show arp",
             "purpose": "ARP テーブルの解決状態を確認。出力が空の場合は直接接続セグメントへの通信がない状態を示す"},
        ],
        "l3": [
            {"cmd": "show route",
             "purpose": "ルーティングテーブル・Inactive ルートを確認"},
            {"cmd": "show bgp summary",
             "purpose": "BGP ピアの状態・受信ルート数を確認。出力が1行のみの場合はネイバー未確立の可能性がある"},
            {"cmd": "show ospf neighbor",
             "purpose": "OSPF ネイバーの状態を確認。出力が空またはネイバーなしの場合はOSPF未設定または隣接未確立"},
        ],
        "recovery_hints": {
            "interface_down": "delete interfaces {ifname} disable  または  set interfaces {ifname} enable",
            "bgp_peer_down":  "show bgp neighbor {peer} detail で詳細確認 → clear bgp neighbor {peer}",
            "route_inactive": "show route {prefix} detail で詳細確認",
            "arp_missing":    "clear arp  でARPキャッシュをクリア後、再確認",
        },
    },

    # ── Cisco IOS / IOS-XE ────────────────────────────────
    # Netmiko driver: cisco_ios
    "cisco_ios": {
        "os_label": "Cisco IOS / IOS-XE",
        "l2": [
            {"cmd": "show interfaces",
             "purpose": "物理インターフェースの Up/Down 状態・エラーカウンタを確認"},
            {"cmd": "show mac address-table",
             "purpose": "MAC アドレステーブルの学習状態・エントリ欠落を確認"},
        ],
        "l3": [
            {"cmd": "show ip route",
             "purpose": "ルーティングテーブル・到達不能ルートを確認"},
            {"cmd": "show ip arp",
             "purpose": "ARP テーブルの解決状態・エントリ欠落を確認"},
        ],
        "recovery_hints": {
            "interface_down": "interface {ifname} → no shutdown",
            "route_inactive": "show ip ospf neighbor / show bgp summary でプロトコル状態を確認",
            "arp_missing":    "clear arp-cache  でARPキャッシュをクリア後、再確認",
        },
    },

    # ── Cisco NX-OS ──────────────────────────────────────
    # Netmiko driver: cisco_nxos
    "cisco_nxos": {
        "os_label": "Cisco NX-OS",
        "l2": [
            {"cmd": "show interface brief",
             "purpose": "全インターフェースの Up/Down 状態を一覧確認"},
            {"cmd": "show mac address-table",
             "purpose": "MAC アドレステーブルの学習状態・エントリ欠落を確認"},
        ],
        "l3": [
            {"cmd": "show ip route",
             "purpose": "ルーティングテーブル・到達不能ルートを確認"},
            {"cmd": "show ip arp",
             "purpose": "ARP テーブルの解決状態・エントリ欠落を確認"},
        ],
        "recovery_hints": {
            "interface_down": "interface {ifname} → no shutdown",
            "route_inactive": "show ip ospf neighbors / show bgp sessions でプロトコル状態を確認",
            "arp_missing":    "clear ip arp force-delete  でARPキャッシュをクリア後、再確認",
        },
    },

    # ── Arista EOS ───────────────────────────────────────
    # Netmiko driver: arista_eos
    "arista_eos": {
        "os_label": "Arista EOS",
        "l2": [
            {"cmd": "show interfaces",
             "purpose": "物理インターフェースの Up/Down 状態・エラーカウンタを確認"},
            {"cmd": "show mac address-table",
             "purpose": "MAC アドレステーブルの学習状態・エントリ欠落を確認"},
        ],
        "l3": [
            {"cmd": "show ip route",
             "purpose": "ルーティングテーブル・到達不能ルートを確認"},
            {"cmd": "show arp",
             "purpose": "ARP テーブルの解決状態・エントリ欠落を確認"},
        ],
        "recovery_hints": {
            "interface_down": "interface {ifname} → no shutdown",
            "route_inactive": "show ip ospf neighbor / show bgp summary でプロトコル状態を確認",
            "arp_missing":    "clear arp-cache  でARPキャッシュをクリア後、再確認",
        },
    },

    # ── 追加機種はここに追記するだけ（コードは変更不要）─────
    # "juniper_qfx": {...},  # Netmiko driver: juniper_junos
    # "cisco_catalyst": {...},  # Netmiko driver: cisco_ios
    # "cisco_asr":      {...},  # Netmiko driver: cisco_ios
    # "nokia_sros":     {...},  # Netmiko driver: nokia_sros
    # "huawei_vrp":     {...},  # Netmiko driver: huawei_vrpv8
}

# ── Netmiko driver マッピング（接続時に参照）─────────────────
# 手順書キー（VENDOR_KEY）→ Netmiko SSH ドライバ
VENDOR_TO_NETMIKO = {
    "juniper_ex":       "juniper_junos",
    "juniper_ptx_evo":  "juniper_junos",   # PTX専用ドライバなし → juniper_junosを使用
    "juniper_mx":       "juniper_junos",
    "cisco_ios":        "cisco_ios",
    "cisco_nxos":       "cisco_nxos",
    "arista_eos":       "arista_eos",
    # 追加時はここにも対応を追記する
}

print("✅ VENDOR_COMMANDS 定義完了")
print(f"   登録機種数: {len(VENDOR_COMMANDS)}")
for vkey, vdata in VENDOR_COMMANDS.items():
    driver = VENDOR_TO_NETMIKO.get(vkey, "（未設定）")
    l2_cmds = [e['cmd'] for e in vdata['l2']]
    l3_cmds = [e['cmd'] for e in vdata['l3']]
    print(f"   📦 {vkey:22s} → Netmiko: {driver}")
    print(f"      L2: {l2_cmds}")
    print(f"      L3: {l3_cmds}")


✅ VENDOR_COMMANDS 定義完了
   登録機種数: 6
   📦 juniper_ex             → Netmiko: juniper_junos
      L2: ['show interfaces', 'show ethernet-switching table']
      L3: ['show route', 'show arp']
   📦 juniper_ptx_evo        → Netmiko: juniper_junos
      L2: ['show interfaces']
      L3: ['show route', 'show arp', 'show bgp summary', 'show ospf neighbor']
   📦 juniper_mx             → Netmiko: juniper_junos
      L2: ['show interfaces', 'show arp']
      L3: ['show route', 'show bgp summary', 'show ospf neighbor']
   📦 cisco_ios              → Netmiko: cisco_ios
      L2: ['show interfaces', 'show mac address-table']
      L3: ['show ip route', 'show ip arp']
   📦 cisco_nxos             → Netmiko: cisco_nxos
      L2: ['show interface brief', 'show mac address-table']
      L3: ['show ip route', 'show ip arp']
   📦 arista_eos             → Netmiko: arista_eos
      L2: ['show interfaces', 'show mac address-table']
      L3: ['show ip route', 'show arp']


## 🗃️ モックデータ（テスト・開発用）

`EXECUTE_MODE = "mock"` のときのみ使用されます。
通常運用では `EXECUTE_MODE = "netmiko"` のため、このセルは実行されません。

### 用途
- 実機なしで動作確認したい場合
- `VENDOR_COMMANDS` に新コマンドを追加した際の単体テスト
- CI/CD環境でのテスト自動化

### 実機データの流用方法
実機から取得した出力をここに貼り替えることで、同じデータで繰り返し分析できます。
```python
MOCK_DATA["show bgp summary"] = """
（実機からコピーした出力をここに貼る）
"""
```


In [7]:
# ============================================================
# テスト・開発用モックデータ
# 通常は EXECUTE_MODE = "netmiko" のため使用されません。
# テスト時は EXECUTE_MODE = "mock" に変更してください。
# ============================================================

# MOCK_DATA = {
#     # ── Juniper PTX EVO 実機データを貼り付ける場合 ────────
#     # "show interfaces": """
#     # （実機からコピーした出力）
#     # """,
#     # "show route": """
#     # （実機からコピーした出力）
#     # """,
#     # "show arp": """
#     # （実機からコピーした出力）
#     # """,
#     # "show bgp summary": """
#     # （実機からコピーした出力）
#     # """,
#     # "show ospf neighbor": """
#     # （実機からコピーした出力）
#     # """,
# }

# netmiko モード用のダミー（EXECUTE_MODE="netmiko" 時は参照されない）
MOCK_DATA = {}

print("✅ モックデータ設定完了（通常運用: 空辞書 / テスト時はコメントを外してデータを追加）")
print(f"   現在の EXECUTE_MODE: {EXECUTE_MODE}")
if EXECUTE_MODE == "mock" and not MOCK_DATA:
    print("   ⚠️  MOCK_DATA が空です。テストする場合はコメントを外してデータを追加してください。")


✅ モックデータ設定完了（通常運用: 空辞書 / テスト時はコメントを外してデータを追加）
   現在の EXECUTE_MODE: netmiko


## 🔌 コマンド実行エンジン

`EXECUTE_MODE` に応じてモック or Netmiko を自動選択します。
エージェント側はこの関数を呼ぶだけで、モード差異を意識しません。


In [8]:
def run_command(command: str) -> Dict[str, str]:
    """
    コマンドを実行してテキスト出力を返す。
    EXECUTE_MODE に応じてモック / Netmiko を自動選択。

    接続時: VENDOR_TO_NETMIKO[VENDOR_KEY] で Netmiko ドライバを解決する。
    → VENDOR_KEY（手順書キー）と Netmiko driver を完全に分離。

    Returns:
        {"command": str, "output": str, "status": "ok" | "error" | "not_found"}
    """
    if EXECUTE_MODE == "mock":
        if command in MOCK_DATA:
            return {"command": command, "output": MOCK_DATA[command], "status": "ok"}
        else:
            return {"command": command, "output": "", "status": "not_found",
                    "message": f"モックデータに '{command}' が登録されていません"}

    elif EXECUTE_MODE == "netmiko":
        try:
            from netmiko import ConnectHandler
            from netmiko.exceptions import (
                AuthenticationException, NetMikoTimeoutException, SSHException
            )
            # VENDOR_KEY → Netmiko driver を解決
            # マッピングにない場合は NETMIKO_DRIVER をフォールバックとして使用
            netmiko_driver = VENDOR_TO_NETMIKO.get(VENDOR_KEY, NETMIKO_DRIVER)
            print(f"  🔌 [{netmiko_driver}] {DEVICE_CONFIG['ip']} に接続中...")
            conn = ConnectHandler(
                device_type=netmiko_driver,
                ip=DEVICE_CONFIG["ip"],
                port=int(DEVICE_CONFIG["port"]),
                username=DEVICE_CONFIG["username"],
                password=DEVICE_CONFIG["password"],
            )
            output = conn.send_command(command, read_timeout=30)
            conn.disconnect()
            return {"command": command, "output": output, "status": "ok"}

        except AuthenticationException:
            return {"command": command, "output": "", "status": "error",
                    "message": "認証失敗: ユーザー名またはパスワードを確認してください"}
        except NetMikoTimeoutException:
            return {"command": command, "output": "", "status": "error",
                    "message": f"接続タイムアウト: {DEVICE_CONFIG['ip']} に到達できません"}
        except SSHException as e:
            return {"command": command, "output": "", "status": "error",
                    "message": f"SSH エラー: {e}"}
        except Exception as e:
            return {"command": command, "output": "", "status": "error",
                    "message": f"予期せぬエラー: {type(e).__name__}: {e}"}
    else:
        return {"command": command, "output": "", "status": "error",
                "message": f"未知の EXECUTE_MODE: {EXECUTE_MODE}"}


def run_commands(commands: List[str]) -> Dict[str, Dict]:
    """複数コマンドをまとめて実行して辞書で返す"""
    results = {}
    for cmd in commands:
        print(f"  ▶ {cmd}", end=" ... ")
        result = run_command(cmd)
        results[cmd] = result
        status_icon = "✅" if result["status"] == "ok" else "❌"
        print(f"{status_icon} ({result['status']})")
    return results


print("✅ run_command() / run_commands() 定義完了")
netmiko_driver = VENDOR_TO_NETMIKO.get(VENDOR_KEY, NETMIKO_DRIVER)
print(f"   実行モード    : {EXECUTE_MODE}")
print(f"   VENDOR_KEY    : {VENDOR_KEY}  →  Netmiko driver: {netmiko_driver}")


✅ run_command() / run_commands() 定義完了
   実行モード    : netmiko
   VENDOR_KEY    : juniper_ptx_evo  →  Netmiko driver: juniper_junos


## 🛠️ エージェントユーティリティ

### 追加機能（テーマ3改良）
| 関数 | 提案 | 内容 |
|------|------|------|
| `display_raw_log()` | 提案1 | 生ログを折り畳みHTMLで表示 |
| `save_command_log()` | 提案3 | コマンド結果を .log ファイルに自動保存 |
| エージェント instructions | 提案2 | `[Evidence]` セクションで根拠引用を強制 |


In [9]:
import os
from datetime import datetime
from IPython.display import display, HTML

def extract_text(response) -> str:
    """Agent レスポンスからテキストを抽出"""
    if hasattr(response, 'text'):
        return str(response.text)
    if hasattr(response, 'messages') and response.messages:
        for msg in response.messages:
            if hasattr(msg, 'text'):
                return str(msg.text)
    return str(response)


def print_section(title: str, width: int = 70):
    print("\n" + "=" * width)
    print(f"  {title}")
    print("=" * width)


def print_agent_result(emoji: str, name: str, text: str):
    print(f"\n{emoji} 【{name}】")
    print("-" * 60)
    for line in text.strip().splitlines():
        print(f"  {line}")


# ── 提案1: 折り畳み表示 ──────────────────────────────────────
def display_raw_log(command: str, output: str, status: str = "ok"):
    """
    コマンド実行結果を折り畳みHTMLで表示する（提案1）。
    AIの分析根拠をエンジニアが即座に目視確認できる。
    """
    if status != "ok" or not output.strip():
        icon = "❌" if status == "error" else "📭"
        label_color = "#f44336" if status == "error" else "#9E9E9E"
        html = f"""
<details>
<summary style="cursor:pointer; color:{label_color}; font-family:monospace;">
  {icon} <b>{command}</b>　（出力なし / {status}）
</summary>
<pre style="background:#fafafa; padding:10px; border-left:3px solid {label_color};
            border-radius:4px; font-size:12px; overflow-x:auto;">（出力なし）</pre>
</details>
"""
    else:
        lines = len(output.strip().splitlines())
        escaped = output.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
        html = f"""
<details>
<summary style="cursor:pointer; color:#1976D2; font-family:monospace;">
  📂 <b>{command}</b>　<span style="color:#757575; font-size:11px;">（{lines} 行 / クリックで生ログ表示）</span>
</summary>
<pre style="background:#f5f5f5; padding:12px; border-left:3px solid #1976D2;
            border-radius:4px; font-size:12px; overflow-x:auto; max-height:400px;">{escaped}</pre>
</details>
"""
    display(HTML(html))


def display_all_raw_logs(cmd_results: dict):
    """全コマンドの生ログを折り畳み表示する"""
    print("\n" + "="*70)
    print("  📂 取得データ一覧（生ログ確認）")
    print("="*70)
    for cmd, res in cmd_results.items():
        display_raw_log(cmd, res.get("output", ""), res.get("status", "error"))


# ── 提案3: 自動ログ保存 ──────────────────────────────────────
LOG_DIR = "logs"

def save_command_log(
    command: str,
    output: str,
    vendor_key: str,
    device_ip: str = "unknown",
    status: str = "ok"
) -> str:
    """
    コマンド結果を .log ファイルに自動保存する（提案3）。
    ファイル名: logs/YYYYMMDD_HHMMSS_{vendor_key}_{ip}_{command}.log
    作業証跡として保管・共有可能。

    Returns:
        保存したファイルパス（失敗時は空文字）
    """
    os.makedirs(LOG_DIR, exist_ok=True)
    ts        = datetime.now().strftime("%Y%m%d_%H%M%S")
    cmd_safe  = command.replace(" ", "_").replace("/", "-")
    ip_safe   = device_ip.replace(".", "-")
    filename  = f"{ts}_{vendor_key}_{ip_safe}_{cmd_safe}.log"
    filepath  = os.path.join(LOG_DIR, filename)
    try:
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(f"# ========================================\n")
            f.write(f"# Timestamp  : {datetime.now().isoformat()}\n")
            f.write(f"# Device IP  : {device_ip}\n")
            f.write(f"# VENDOR_KEY : {vendor_key}\n")
            f.write(f"# Command    : {command}\n")
            f.write(f"# Status     : {status}\n")
            f.write(f"# ========================================\n\n")
            f.write(output if output else "(no output)")
        return filepath
    except Exception as e:
        print(f"  ⚠️  ログ保存失敗: {e}")
        return ""


def save_all_logs(cmd_results: dict, vendor_key: str, device_ip: str) -> list:
    """全コマンドのログを保存してファイルパス一覧を返す"""
    saved = []
    for cmd, res in cmd_results.items():
        path = save_command_log(
            command=cmd,
            output=res.get("output", ""),
            vendor_key=vendor_key,
            device_ip=device_ip,
            status=res.get("status", "error")
        )
        if path:
            saved.append(path)
    return saved


print("✅ ユーティリティ定義完了")
print("   提案1: display_raw_log() / display_all_raw_logs()  → 折り畳み生ログ表示")
print("   提案3: save_command_log() / save_all_logs()        → .log ファイル自動保存")


✅ ユーティリティ定義完了
   提案1: display_raw_log() / display_all_raw_logs()  → 折り畳み生ログ表示
   提案3: save_command_log() / save_all_logs()        → .log ファイル自動保存


## 🏗️ NetworkDiagnosticSystem クラス（本体）

5つの専門エージェントが連携します。

| エージェント | 役割 | 参照データ |
|-------------|------|-----------|
| コマンド決定 | 状況から必要コマンドを判断 | ユーザー入力 |
| L2分析 | インターフェース・MAC異常の特定 | show interfaces, show mac |
| L3分析 | ルーティング・ARP異常の特定 | show route, show arp |
| 整合性チェック | L2/L3の矛盾を検出・Self-Correction | L2+L3両結果 |
| 診断レポート | 根本原因と推奨アクションを出力 | 全エージェント結果 |


In [10]:
class NetworkDiagnosticSystem:
    """
    マルチレイヤ障害診断エージェントシステム（マルチベンダー対応版）

    役割分担:
      YAML辞書 → コマンド定義・purpose・recovery_hints（構造化情報）
      LLM      → フロー遷移判断・出力解釈・整合性チェック・レポート生成
    """

    def __init__(self, model: str = DEFAULT_MODEL):
        self.model = model
        self._init_agents()

    def _init_agents(self):
        """5つの専門エージェントを初期化（instructions はフロー系のみ）"""
        print("\n🔧 エージェント初期化中...")

        # ① フロー遷移判断エージェント（LLMの役割: l2/l3/full の選択）
        self.cmd_agent = Agent(
            name="フロー遷移判断",
            client=make_client(self.model),
            instructions=(
                "あなたはネットワーク障害診断のトリアージ専門家です。"
                "ユーザーの状況説明を読み、診断フローを選択してください。"
                "以下のいずれかを **1単語だけ** 出力してください。他の文字は出力禁止です。\n"
                "- l2   : L2層のみ（インターフェース・MAC障害が明確に疑われる場合）\n"
                "- l3   : L3層のみ（ルーティング・到達性障害が明確に疑われる場合）\n"
                "- full : L2/L3両方（原因不明・複合障害・定期確認の場合）"
            ),
        )

        # ② L2分析エージェント（instructions はフロー系のみ・コマンド名はpromptで注入）
        self.l2_agent = Agent(
            name="L2分析",
            client=make_client(self.model),
            instructions=(
                "あなたは L2 層（データリンク層）の専門分析エージェントです。\n"
                "提供されたコマンド出力を分析し、以下の観点で異常を特定してください:\n"
                "- インターフェースの Up/Down 状態\n"
                "- インターフェースエラーカウンタ（Input errors / CRC / resets など）\n"
                "- MAC アドレステーブルの不整合（期待されるエントリの欠落など）\n"
                "\n"
                "【重要】回答は必ず以下の形式で出力してください:\n"
                "\n"
                "[分析結果]\n"
                "異常点を箇条書きで。正常な場合は「L2 層に異常なし」と述べる。\n"
                "\n"
                "[Evidence]\n"
                "各判断の根拠となった生ログの該当箇所を必ず引用する。\n"
                "例: et-0/0/2 Down の根拠 → 'Physical link is Down'（show interfaces より）\n"
                "推測ではなく、生ログに記載された事実のみを引用すること。\n"
                "\n"
                "日本語で回答してください。"
            ),
        )

        # ③ L3分析エージェント
        self.l3_agent = Agent(
            name="L3分析",
            client=make_client(self.model),
            instructions=(
                "あなたは L3 層（ネットワーク層）の専門分析エージェントです。\n"
                "提供されたコマンド出力を分析し、以下の観点で異常を特定してください:\n"
                "- Inactive ルート・ルートの欠落\n"
                "- ARP エントリの欠落（通信できるはずのホストが ARP にない）\n"
                "- デフォルトルートや静的ルートの異常\n"
                "- BGP ピアの状態・受信ルート数の異常\n"
                "- OSPF ネイバーの確立状態\n"
                "\n"
                "【重要】回答は必ず以下の形式で出力してください:\n"
                "\n"
                "[分析結果]\n"
                "異常点を箇条書きで。正常な場合は「L3 層に異常なし」と述べる。\n"
                "\n"
                "[Evidence]\n"
                "各判断の根拠となった生ログの該当箇所を必ず引用する。\n"
                "例: BGP Active の根拠 → '192.168.1.1  Active  0  0  0'（show bgp summary より）\n"
                "推測ではなく、生ログに記載された事実のみを引用すること。\n"
                "\n"
                "日本語で回答してください。"
            ),
        )

        # ④ 整合性チェックエージェント（Self-Correction）
        self.consistency_agent = Agent(
            name="整合性チェック",
            client=make_client(self.model),
            instructions=(
                "あなたは L2/L3 整合性の専門チェックエージェントです。\n"
                "L2 分析結果と L3 分析結果を突き合わせ、以下のような矛盾を検出してください:\n"
                "- L2 でインターフェースがダウン → L3 でそのインターフェース経由のルートが Inactive\n"
                "- MAC テーブルにエントリあり → ARP テーブルに対応エントリがない\n"
                "- L3 でルートが消失 → L2 のインターフェースダウンが原因か否か\n"
                "検出した矛盾・因果関係を箇条書きで。"
                "矛盾がなければ「L2/L3 間に整合性の問題なし」と述べてください。"
                "日本語で回答してください。"
            ),
        )

        # ⑤ 診断レポートエージェント（recovery_hints は diagnose() で動的注入）
        self.report_agent = Agent(
            name="診断レポート",
            client=make_client(self.model),
            instructions=(
                "あなたはネットワーク障害診断の最終レポート作成エージェントです。\n"
                "L2分析・L3分析・整合性チェックの結果と、OS固有の復旧ヒントを統合し、\n"
                "以下の構成でレポートを作成してください:\n\n"
                "【根本原因】\n"
                "  最も根本的な障害原因を1〜2文で断定的に述べる\n\n"
                "【影響範囲】\n"
                "  影響を受けているサービス・通信を箇条書きで列挙\n\n"
                "【推奨アクション】\n"
                "  提供された OS 固有の復旧ヒントを参照しながら、\n"
                "  優先順位順に具体的な手順を箇条書きで列挙\n\n"
                "日本語で回答してください。"
            ),
        )

        print("  ✓ 🎯 フロー遷移判断エージェント（LLM）")
        print("  ✓ 🔵 L2 分析エージェント（LLM）")
        print("  ✓ 🟢 L3 分析エージェント（LLM）")
        print("  ✓ ⚖️  整合性チェックエージェント（LLM / Self-Correction）")
        print("  ✓ 📄 診断レポートエージェント（LLM + YAML recovery_hints）")
        print(f"\n✅ 全エージェント初期化完了")

    def _get_vendor_config(self, vendor_key: str) -> Dict:
        """YAML辞書から指定 VENDOR_KEY のコマンド定義を取得"""
        if vendor_key not in VENDOR_COMMANDS:
            supported = list(VENDOR_COMMANDS.keys())
            raise ValueError(
                f"未対応の VENDOR_KEY: '{vendor_key}'\n"
                f"対応済み: {supported}\n"
                f"VENDOR_COMMANDS に定義を追加してください。"
            )
        return VENDOR_COMMANDS[vendor_key]

    def _build_command_context(self, entries: List[Dict], cmd_results: Dict) -> str:
        """
        コマンド出力に purpose を付与してLLMへの文脈情報を構築する。
        YAMLのpurposeをプロンプトに注入することで分析精度を向上させる。
        """
        sections = []
        for entry in entries:
            cmd = entry["cmd"]
            purpose = entry["purpose"]
            if cmd in cmd_results and cmd_results[cmd]["status"] == "ok":
                sections.append(
                    f"## {cmd}\n"
                    f"（確認目的: {purpose}）\n"
                    f"{cmd_results[cmd]['output']}"
                )
        return "\n\n".join(sections)

    async def diagnose(self, user_input: str, vendor_key: str = None) -> Dict[str, Any]:
        """
        障害診断を実行する。

        Args:
            user_input  : ユーザーからの状況説明
            device_type : ネットワーク OS 種別（省略時は DEVICE_TYPE グローバル変数を使用）

        Returns:
            diagnosis_result: 各エージェントの分析結果を含む辞書
        """
        vendor_key = vendor_key or VENDOR_KEY
        started_at = datetime.now()

        # YAML辞書からOS設定を取得（コマンドはLLMが生成しない）
        vendor_cfg = self._get_vendor_config(vendor_key)
        os_label   = vendor_cfg["os_label"]
        netmiko_driver = VENDOR_TO_NETMIKO.get(vendor_key, NETMIKO_DRIVER)

        result = {
            "user_input":        user_input,
            "vendor_key":        vendor_key,
            "netmiko_driver":    netmiko_driver,
            "os_label":          os_label,
            "execute_mode":      EXECUTE_MODE,
            "flow":              None,
            "commands_executed": {},
            "l2_analysis":       None,
            "l3_analysis":       None,
            "consistency":       None,
            "report":            None,
            "started_at":        started_at.isoformat(),
        }

        print_section(
            f"🔍 ネットワーク障害診断開始  "
            f"[{EXECUTE_MODE.upper()} / {os_label}]"
        )
        print(f"\n  入力: {user_input}")

        # ── Step 0: LLM がフロー遷移を判断 ───────────────────
        print_section("Step 0: フロー遷移判断（LLM）")
        resp = await self.cmd_agent.run(
            f"以下の状況に対して診断フローを選択してください。\n\n{user_input}"
        )
        flow_raw = extract_text(resp).strip().lower().split()[0]
        flow = flow_raw if flow_raw in ("l2", "l3", "full") else "full"
        result["flow"] = flow
        print(f"  🎯 選択されたフロー: [{flow}]")

        # ── Step 1: YAML辞書からコマンドを取得して実行 ────────
        print_section(f"Step 1: コマンド実行（{EXECUTE_MODE} / YAML辞書参照）")
        l2_entries = vendor_cfg["l2"] if flow in ("l2", "full") else []
        l3_entries = vendor_cfg["l3"] if flow in ("l3", "full") else []
        all_entries = l2_entries + l3_entries

        print(f"  📖 {os_label} のコマンド辞書を参照")
        cmd_results = run_commands([e["cmd"] for e in all_entries])
        result["commands_executed"] = cmd_results

        # ── 提案1: 折り畳み生ログ表示 ────────────────────────
        display_all_raw_logs(cmd_results)

        # ── 提案3: ログファイル自動保存 ──────────────────────
        device_ip = DEVICE_CONFIG.get("ip", "unknown") if EXECUTE_MODE == "netmiko" else "mock"
        saved_logs = save_all_logs(cmd_results, vendor_key, device_ip)
        result["saved_logs"] = saved_logs
        if saved_logs:
            print(f"\n  💾 ログ保存完了: {len(saved_logs)} ファイル → {LOG_DIR}/")
            for p in saved_logs:
                print(f"     {p}")

        # ── Step 2: L2分析（purpose をプロンプトに注入）────────
        if l2_entries:
            print_section(f"Step 2: L2 分析（{os_label}）")
            l2_context = self._build_command_context(l2_entries, cmd_results)
            resp = await self.l2_agent.run(
                f"ネットワーク OS: {os_label}\n\n"
                f"以下の L2 コマンド出力を分析してください。\n\n{l2_context}"
            )
            result["l2_analysis"] = extract_text(resp)
            print_agent_result("🔵", "L2 分析エージェント", result["l2_analysis"])
        else:
            result["l2_analysis"] = "（L2 コマンドは今回のフローでスキップ）"

        # ── Step 3: L3分析（purpose をプロンプトに注入）────────
        if l3_entries:
            print_section(f"Step 3: L3 分析（{os_label}）")
            l3_context = self._build_command_context(l3_entries, cmd_results)
            resp = await self.l3_agent.run(
                f"ネットワーク OS: {os_label}\n\n"
                f"以下の L3 コマンド出力を分析してください。\n\n{l3_context}"
            )
            result["l3_analysis"] = extract_text(resp)
            print_agent_result("🟢", "L3 分析エージェント", result["l3_analysis"])
        else:
            result["l3_analysis"] = "（L3 コマンドは今回のフローでスキップ）"

        # ── Step 4: 整合性チェック（Self-Correction）────────
        print_section("Step 4: 整合性チェック（Self-Correction）")
        resp = await self.consistency_agent.run(
            f"【L2 分析結果】\n{result['l2_analysis']}\n\n"
            f"【L3 分析結果】\n{result['l3_analysis']}"
        )
        result["consistency"] = extract_text(resp)
        print_agent_result("⚖️ ", "整合性チェックエージェント", result["consistency"])

        if any(kw in result["consistency"] for kw in ("矛盾", "因果", "Inactive", "ダウン")):
            print("\n  🔄 [Self-Correction] L3 エージェントが原因を再確認中...")
            resp2 = await self.l3_agent.run(
                f"整合性チェックで以下の矛盾が指摘されました:\n{result['consistency']}\n\n"
                f"あなたの元の L3 分析: {result['l3_analysis']}\n\n"
                f"ネットワーク OS: {os_label}\n"
                f"指摘を踏まえて、L2 障害と L3 異常の因果関係を明確にして再分析してください。"
            )
            result["l3_analysis_corrected"] = extract_text(resp2)
            print_agent_result("🟢", "L3 分析エージェント【修正後】",
                               result["l3_analysis_corrected"])

        # ── Step 5: 診断レポート（YAML recovery_hints を注入）─
        print_section("Step 5: 最終診断レポート")

        # recovery_hints を OS 固有のコマンド例としてプロンプトに注入
        hints = vendor_cfg["recovery_hints"]
        hints_text = "\n".join(
            f"  - {k}: {v}" for k, v in hints.items()
        )

        resp = await self.report_agent.run(
            f"【ユーザー報告】\n{user_input}\n\n"
            f"【ネットワーク OS】\n{os_label}\n\n"
            f"【L2 分析結果】\n{result['l2_analysis']}\n\n"
            f"【L3 分析結果】\n"
            f"{result.get('l3_analysis_corrected', result['l3_analysis'])}\n\n"
            f"【整合性チェック結果】\n{result['consistency']}\n\n"
            f"【OS 固有の復旧ヒント（推奨アクションのコマンド例に使用すること）】\n"
            f"{hints_text}"
        )
        result["report"] = extract_text(resp)
        print_agent_result("📄", "診断レポートエージェント", result["report"])

        elapsed = (datetime.now() - started_at).total_seconds()
        print_section(f"✅ 診断完了  （所要時間: {elapsed:.1f}秒）")
        result["elapsed_seconds"] = elapsed
        return result

print("✅ NetworkDiagnosticSystem クラス定義完了（マルチベンダー対応版）")


✅ NetworkDiagnosticSystem クラス定義完了（マルチベンダー対応版）


## 🎬 診断シナリオ設定

`USER_INPUT` を変えると、コマンド決定エージェントが必要なコマンドセットを自動選択します。

```python
# 例1: 原因不明 → full セットが選ばれるはず
"社内から「インターネットにつながらない」という報告が複数来ています。"

# 例2: L2 疑い → l2 セットが選ばれるはず
"et-0/0/2 のランプが消えています。物理的な問題か確認したい。"

# 例3: L3 疑い → l3 セットが選ばれるはず
"ping は通るのにルーティングがおかしい。特定のサブネットに届かない。"
```


In [11]:
# ============================================================
# ✏️ ここを変えてください
# ============================================================

# 障害申告の内容
USER_INPUT = (
    "社内の一部ユーザーからインターネット接続が不安定という報告があります。"
    "また、別の拠点 (192.168.20.0/24) との通信が完全に切れています。"
    "ネットワーク全体の正常性を確認してください。"
)

# ✏️ VENDOR_KEY: 手順書由来のコマンド辞書キー
#    VENDOR_COMMANDS に定義されているキーを指定する
#    例: "juniper_ptx_evo" / "juniper_ex" / "cisco_ios" / "cisco_nxos" など
SCENARIO_VENDOR_KEY = "juniper_ptx_evo"

print("✅ シナリオ設定完了")
print(f"   VENDOR_KEY  : {SCENARIO_VENDOR_KEY}")
driver = VENDOR_TO_NETMIKO.get(SCENARIO_VENDOR_KEY, NETMIKO_DRIVER)
print(f"   Netmiko driver: {driver}  （自動解決）")
print(f"   ユーザー入力: {USER_INPUT}")


✅ シナリオ設定完了
   VENDOR_KEY  : juniper_ptx_evo
   Netmiko driver: juniper_junos  （自動解決）
   ユーザー入力: 社内の一部ユーザーからインターネット接続が不安定という報告があります。また、別の拠点 (192.168.20.0/24) との通信が完全に切れています。ネットワーク全体の正常性を確認してください。


## 🚀 診断実行

In [12]:
# 診断システムを初期化して実行
diag = NetworkDiagnosticSystem()
diagnosis_result = await diag.diagnose(
    user_input=USER_INPUT,
    vendor_key=SCENARIO_VENDOR_KEY,
)



🔧 エージェント初期化中...
  ✓ 🎯 フロー遷移判断エージェント（LLM）
  ✓ 🔵 L2 分析エージェント（LLM）
  ✓ 🟢 L3 分析エージェント（LLM）
  ✓ ⚖️  整合性チェックエージェント（LLM / Self-Correction）
  ✓ 📄 診断レポートエージェント（LLM + YAML recovery_hints）

✅ 全エージェント初期化完了

  🔍 ネットワーク障害診断開始  [NETMIKO / Juniper PTX Series (Core Router / Junos EVO)]

  入力: 社内の一部ユーザーからインターネット接続が不安定という報告があります。また、別の拠点 (192.168.20.0/24) との通信が完全に切れています。ネットワーク全体の正常性を確認してください。

  Step 0: フロー遷移判断（LLM）
  🎯 選択されたフロー: [full]

  Step 1: コマンド実行（netmiko / YAML辞書参照）
  📖 Juniper PTX Series (Core Router / Junos EVO) のコマンド辞書を参照
  ▶ show interfaces ...   🔌 [juniper_junos] 172.20.100.21 に接続中...
✅ (ok)
  ▶ show route ...   🔌 [juniper_junos] 172.20.100.21 に接続中...
✅ (ok)
  ▶ show arp ...   🔌 [juniper_junos] 172.20.100.21 に接続中...
✅ (ok)
  ▶ show bgp summary ...   🔌 [juniper_junos] 172.20.100.21 に接続中...
✅ (ok)
  ▶ show ospf neighbor ...   🔌 [juniper_junos] 172.20.100.21 に接続中...
✅ (ok)

  📂 取得データ一覧（生ログ確認）



  💾 ログ保存完了: 5 ファイル → logs/
     logs/20260320_032752_juniper_ptx_evo_172-20-100-21_show_interfaces.log
     logs/20260320_032752_juniper_ptx_evo_172-20-100-21_show_route.log
     logs/20260320_032752_juniper_ptx_evo_172-20-100-21_show_arp.log
     logs/20260320_032752_juniper_ptx_evo_172-20-100-21_show_bgp_summary.log
     logs/20260320_032752_juniper_ptx_evo_172-20-100-21_show_ospf_neighbor.log

  Step 2: L2 分析（Juniper PTX Series (Core Router / Junos EVO)）

🔵 【L2 分析エージェント】
------------------------------------------------------------
  [分析結果]
  L2 層に異常なし
  
  [Evidence]
  - 全てのインターフェースの Physical link は "Up" であることを確認しました（例: "Physical interface: et-0/0/0, Enabled, Physical link is Up"）。
  - エラーカウンタ（Bit errors、Errored blocksなど）は全て0であることを確認しました（例: "PCS statistics                      Seconds Bit errors                             0 Errored blocks                         0"）。
  - MAC アドレステーブルの不整合は見られませんでした。全てのインターフェースに割り当てられたMACアドレスは一意であることを確認しました（例: "Current address: 4a:09:00:1c:ff:ff, Ha

## 📊 診断結果サマリー

In [13]:
print("\n" + "="*70)
print("📊 診断結果サマリー")
print("="*70)

print(f"\n  実行モード     : {diagnosis_result['execute_mode'].upper()}")
print(f"  ネットワーク OS : {diagnosis_result['os_label']}")
print(f"  VENDOR_KEY     : {diagnosis_result['vendor_key']}  （手順書キー）")
print(f"  Netmiko driver : {diagnosis_result['netmiko_driver']}  （SSH接続ドライバ）")
print(f"  選択フロー     : {diagnosis_result['flow']}")
print(f"  所要時間       : {diagnosis_result.get('elapsed_seconds', 0):.1f} 秒")

print("\n  【実行コマンド（YAML辞書より取得）】")
for cmd, res in diagnosis_result["commands_executed"].items():
    lines = len(res["output"].strip().splitlines()) if res["output"] else 0
    status_icon = "✅" if res["status"] == "ok" else "❌"
    print(f"  {status_icon} {cmd} ({lines} 行取得)")

if "l3_analysis_corrected" in diagnosis_result:
    print("\n  ⚡ Self-Correction: 発動（L2/L3 矛盾を検出 → L3 エージェントが再分析）")
else:
    print("\n  ✅ Self-Correction: 発動なし（L2/L3 整合性に問題なし）")

# ── 提案3: 保存ログ一覧 ─────────────────────────────────
saved = diagnosis_result.get("saved_logs", [])
if saved:
    print("\n" + "="*70)
    print("💾 保存済みログファイル一覧")
    print("="*70)
    for p in saved:
        print(f"  📄 {p}")
    print("\n  ※ JupyterLab のファイルツリー（左パネル）→ logs/ から参照できます")

print("\n" + "="*70)
print("📄 最終診断レポート")
print("="*70)
print(diagnosis_result["report"])

# 📂 生ログ折り畳み表示（サマリーセルでも確認できるように再掲）
from IPython.display import display, HTML
display(HTML("<hr><b>📂 生ログ（クリックして展開）</b>"))
display_all_raw_logs(diagnosis_result["commands_executed"])



📊 診断結果サマリー

  実行モード     : NETMIKO
  ネットワーク OS : Juniper PTX Series (Core Router / Junos EVO)
  VENDOR_KEY     : juniper_ptx_evo  （手順書キー）
  Netmiko driver : juniper_junos  （SSH接続ドライバ）
  選択フロー     : full
  所要時間       : 8.7 秒

  【実行コマンド（YAML辞書より取得）】
  ✅ show interfaces (559 行取得)
  ✅ show route (98 行取得)
  ✅ show arp (0 行取得)
  ✅ show bgp summary (10 行取得)
  ✅ show ospf neighbor (1 行取得)

  ✅ Self-Correction: 発動なし（L2/L3 整合性に問題なし）

💾 保存済みログファイル一覧
  📄 logs/20260320_032752_juniper_ptx_evo_172-20-100-21_show_interfaces.log
  📄 logs/20260320_032752_juniper_ptx_evo_172-20-100-21_show_route.log
  📄 logs/20260320_032752_juniper_ptx_evo_172-20-100-21_show_arp.log
  📄 logs/20260320_032752_juniper_ptx_evo_172-20-100-21_show_bgp_summary.log
  📄 logs/20260320_032752_juniper_ptx_evo_172-20-100-21_show_ospf_neighbor.log

  ※ JupyterLab のファイルツリー（左パネル）→ logs/ から参照できます

📄 最終診断レポート
【根本原因】
社内の一部ユーザーからインターネット接続が不安定であることと、別の拠点 (192.168.20.0/24) との通信が完全に切れていることは、BGP ピアの Down peers のカウントが 1 であることと、ライセンスキーが Missing で


  📂 取得データ一覧（生ログ確認）


---
## 🔄 実機モードへの切り替え方法

実機テストの準備ができたら、以下の2ステップだけで切り替えられます。

### Step 1: モードとデバイス設定を変更（Cell 5）

```python
EXECUTE_MODE = "netmiko"   # "mock" → "netmiko" に変更

DEVICE_CONFIG = {
    "ip":          "172.20.100.21",   # ← 実機のIPに変更
    "port":        "22",
    "username":    "admin",
    "password": "YOUR_PASSWORD_HERE",
    "device_type": "juniper_junos",
}
```

### Step 2: 「カーネル → すべてのセルを再実行」

エージェント・モックデータ・診断ロジックは一切変更不要です。

---

## 📋 対応コマンドの追加方法

新しいコマンドを追加したい場合は2箇所だけ変更します。

```python
# 1. MOCK_DATA に追加（Cell 6）
MOCK_DATA["show ospf neighbor"] = """
Address          Interface              State      ...
10.0.0.1         et-0/0/0.0             Full       ...
"""

# 2. COMMAND_SETS に追加（NetworkDiagnosticSystem クラス内）
COMMAND_SETS = {
    "l3": ["show route", "show arp", "show ospf neighbor"],  # ← 追加
    ...
}
```


---
## 🗺️ 今後の拡張ロードマップ

| フェーズ | 内容 | 対応箇所 |
|----------|------|---------|
| **完成済み** | VENDOR_KEY / NETMIKO_DRIVER 分離でマルチベンダー対応 | — |
| **次回** | `EXECUTE_MODE = "netmiko"` で実機接続 | Cell 10 の1行 |
| **発展1** | VENDOR_COMMANDS を外部 YAML ファイルに外出し | `yaml.safe_load()` で読み込み |
| **発展2** | BGP/OSPF 専門エージェントを追加 | `juniper_ptx_evo` に l4 セクション追加 |
| **発展3** | Phase 6 Orchestrator の Skill として統合 | `diagnose()` を Skill 登録 |

### VENDOR_KEY と Netmiko driver の対応表（VENDOR_TO_NETMIKO）

| VENDOR_KEY | Netmiko driver | 機種 |
|-----------|---------------|------|
| `juniper_ex` | `juniper_junos` | Juniper EX（L2スイッチ）|
| `juniper_ptx_evo` | `juniper_junos` | Juniper PTX（コアルータ / EVO）|
| `juniper_mx` | `juniper_junos` | Juniper MX（エッジルータ）|
| `cisco_ios` | `cisco_ios` | Cisco IOS / IOS-XE |
| `cisco_nxos` | `cisco_nxos` | Cisco NX-OS |
| `arista_eos` | `arista_eos` | Arista EOS |

### 外部 YAML ファイルに外出しする場合
```
vendor_commands/
  juniper_ex.yaml
  juniper_ptx_evo.yaml   ← 追加するだけ
  juniper_mx.yaml
  cisco_ios.yaml
  ...
```
```python
import yaml, glob
VENDOR_COMMANDS = {}
for path in glob.glob("vendor_commands/*.yaml"):
    key = path.stem  # ファイル名がそのまま VENDOR_KEY
    with open(path) as f:
        VENDOR_COMMANDS[key] = yaml.safe_load(f)
```
